In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

print("✅ Librerías cargadas")
print(f"Directorio actual: {os.getcwd()}")

✅ Librerías cargadas
Directorio actual: /workspaces/ECONOMIA-DEL-CUIDADO/notebooks


In [4]:
# Cargar los datos
enut = pd.read_csv('../data/raw/ENUT_2024_CDMX_base_preliminar.csv')
resumen = pd.read_csv('../data/raw/ENUT_2024_CDMX_resumen_ponderado.csv')
censo = pd.read_csv('../data/raw/base_censo_CDMX_observatorio_v1.csv')

print(f"ENUT: {enut.shape}")
print(f"Resumen: {resumen.shape}")
print(f"Censo: {censo.shape}")

ENUT: (2107, 29)
Resumen: (2, 13)
Censo: (16, 43)


In [5]:
print("Columnas de la ENUT:")
for col in enut.columns:
    print(f"  - {col}")

Columnas de la ENUT:
  - llavemod
  - llaveviv
  - llavehog
  - cve_ent
  - cvegeo
  - edad
  - sexo
  - sexo_label
  - escolaridad
  - cond_disc
  - cond_disc_label
  - cond_aee
  - trab_no_rem_cuid_hog
  - cuid_esp_int_hog_sin_cp
  - cuid_int_0a5_sin_cp
  - cuid_int_6a14_sin_cp
  - cuid_int_15a59
  - cuid_int_60mas_sin_cp
  - trab_merc_pv
  - bus_trab
  - fac_per
  - realiza_cuidado_hogar
  - realiza_cuidado_especial
  - realiza_cuidado_0a5
  - realiza_cuidado_6a14
  - realiza_cuidado_15a59
  - realiza_cuidado_60mas
  - realiza_trabajo_mercado
  - busca_trabajo


In [6]:
enut_15 = enut[enut['edad'] >= 15].copy()
print(f"Registros originales: {len(enut):,}")
print(f"Registros 15+: {len(enut_15):,}")

Registros originales: 2,107
Registros 15+: 2,040


In [ ]:
VARIABLES DERIVADAS


In [7]:
enut_15['es_cuidador'] = (enut_15['realiza_cuidado_hogar'] == 1).astype(int)
enut_15['horas_cuidado'] = enut_15['trab_no_rem_cuid_hog'].fillna(0)
enut_15['horas_trabajo'] = enut_15['trab_merc_pv'].fillna(0)
enut_15['participa_laboral'] = (enut_15['trab_merc_pv'] > 0).astype(int)
enut_15['combina_cuidado_trabajo'] = (
    (enut_15['es_cuidador'] == 1) & (enut_15['participa_laboral'] == 1)
).astype(int)

print("✅ Variables creadas")

✅ Variables creadas


In [ ]:
LIMPIAR VALORES EXTREMOS

In [8]:
enut_15 = enut_15[enut_15['horas_cuidado'] <= 168]
enut_15 = enut_15[enut_15['horas_trabajo'] <= 168]
print(f"Registros después de limpieza: {len(enut_15):,}")

Registros después de limpieza: 2,040


In [ ]:
GUARDAR LAS BASES PROCESADAS

In [9]:
enut_15.to_csv('../data/processed/enut_limpia.csv', index=False)

cuidadores = enut_15[enut_15['es_cuidador'] == 1].copy()
cuidadores.to_csv('../data/processed/enut_cuidadores.csv', index=False)

print("✅ Archivos guardados en data/processed/")
print(f"  📄 enut_limpia.csv: {enut_15.shape[0]:,} filas")
print(f"  📄 enut_cuidadores.csv: {cuidadores.shape[0]:,} filas")

✅ Archivos guardados en data/processed/
  📄 enut_limpia.csv: 2,040 filas
  📄 enut_cuidadores.csv: 1,237 filas


In [ ]:
RESUMEN FINAL

In [10]:
print("=" * 50)
print("RESUMEN DE LIMPIEZA")
print("=" * 50)
print(f"Total 15+: {len(enut_15):,}")
print(f"Cuidadores: {cuidadores.shape[0]:,}")
print(f"Trabajan: {enut_15['participa_laboral'].sum():,}")
print(f"\nDistribución por sexo:")
print(enut_15['sexo_label'].value_counts())
print(f"\nCuidadores por sexo:")
print(cuidadores['sexo_label'].value_counts())

RESUMEN DE LIMPIEZA
Total 15+: 2,040
Cuidadores: 1,237
Trabajan: 1,294

Distribución por sexo:
sexo_label
Mujer     1114
Hombre     926
Name: count, dtype: int64

Cuidadores por sexo:
sexo_label
Mujer     688
Hombre    549
Name: count, dtype: int64
